<a href="https://colab.research.google.com/github/nerd2ninja/Colab-notebooks/blob/main/whisperx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install whisperx

In [ ]:
pip uninstall -y torch torchvision torchaudio

pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \
  --index-url https://download.pytorch.org/whl/cu118

pip install whisperx==3.1.1 transformers==4.41.2

In [ ]:
from google.colab import files
uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

In [ ]:
import whisperx
import torch
import json

device = "cuda"
compute_type = "float16"

# 🔥 High-quality model
model = whisperx.load_model("large-v2", device, compute_type=compute_type)

# Step 1: Transcribe
result = model.transcribe(audio_file)

# Step 2: Align (word-level precision)
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"],
    device=device
)

result = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio_file,
    device
)

# Step 3: Build clean sentence segments
sentence_segments = []

for seg in result["segments"]:
    words = seg.get("words", [])
    if not words:
        continue

    sentence_segments.append({
        "start": words[0]["start"],
        "end": words[-1]["end"],
        "text": seg["text"].strip()
    })

# Print preview
print(json.dumps(sentence_segments[:5], indent=2, ensure_ascii=False))